In [1]:
import re
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PATH = "../data/names.txt"

In [3]:
with open(PATH, encoding="utf-8") as f:
    names = [line.strip() for line in f if line.strip()]

# Sadece harflerden oluşan isimleri al, küçük harfe çevir, tekrarları kaldır
names = [n.lower() for n in names if re.fullmatch(r"[A-Za-z]+", n)]
names = sorted(set(names))
random.shuffle(names)
print("Temiz isim sayısı:", len(names))

Temiz isim sayısı: 33440


In [4]:
PAD, START, END = "<PAD>", "^", "$"
chars = sorted(set("".join(names)))
itos = [PAD, START, END] + chars
stoi = {ch: i for i, ch in enumerate(itos)}
VOCAB = len(itos)
print("Vocabulary boyutu:", VOCAB)

Vocabulary boyutu: 29


In [5]:
MAX_LEN = max(len(n) for n in names) + 1

X_all, y_all = [], []
for n in names:
    ids = [stoi[START]] + [stoi[c] for c in n] + [stoi[END]]
    X_all.append(ids[:-1])
    y_all.append(ids[1:])

X_all = pad_sequences(X_all, maxlen=MAX_LEN, padding="post", value=0)
y_all = pad_sequences(y_all, maxlen=MAX_LEN, padding="post", value=0)

In [6]:
n_val = int(0.1 * len(X_all))
X_val, y_val = X_all[:n_val], y_all[:n_val]
X_train, y_train = X_all[n_val:], y_all[n_val:]
print("Train:", len(X_train), "Val:", len(X_val))

Train: 30096 Val: 3344


In [7]:
model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB, 32, mask_zero=True),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.2),
    layers.Dense(VOCAB, activation="softmax"),
])

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 16, 32)            928       
                                                                 
 lstm (LSTM)                 (None, 16, 128)           82432     
                                                                 
 dropout (Dropout)           (None, 16, 128)           0         
                                                                 
 lstm_1 (LSTM)               (None, 16, 128)           131584    
                                                                 
 dropout_1 (Dropout)         (None, 16, 128)           0         
                                                                 
 dense (Dense)               (None, 16, 29)            3741      
                                                                 
Total params: 218,685
Trainable params: 218,685
Non-trai

In [9]:
es = callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

In [10]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[es],
    verbose=1,
)

Epoch 1/50
471/471 [==============================] - 24s 36ms/step - loss: 1.1612 - val_loss: 1.0685
Epoch 2/50
471/471 [==============================] - 16s 33ms/step - loss: 1.0548 - val_loss: 1.0178
Epoch 3/50
471/471 [==============================] - 16s 34ms/step - loss: 1.0169 - val_loss: 0.9936
Epoch 4/50
471/471 [==============================] - 17s 37ms/step - loss: 0.9954 - val_loss: 0.9764
Epoch 5/50
471/471 [==============================] - 18s 37ms/step - loss: 0.9801 - val_loss: 0.9618
Epoch 6/50
471/471 [==============================] - 17s 37ms/step - loss: 0.9670 - val_loss: 0.9506
Epoch 7/50
471/471 [==============================] - 18s 37ms/step - loss: 0.9571 - val_loss: 0.9404
Epoch 8/50
471/471 [==============================] - 22s 46ms/step - loss: 0.9470 - val_loss: 0.9337
Epoch 9/50
471/471 [==============================] - 22s 47ms/step - loss: 0.9384 - val_loss: 0.9264
Epoch 10/50
471/471 [==============================] - 23s 48ms/step - loss: 0.930

In [12]:
model.save("../model/name_lstm.keras")